<a href="https://colab.research.google.com/github/phineas-pta/gg_colab_AI_playground/blob/main/trans_ZH_VI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# convert thô nhanh = AI

sử dụng model:
- https://huggingface.co/ngocdang83/HachimiMT-60-zh-vi
- https://huggingface.co/DanVP/MoxhiMT-60

In [ ]:
#@markdown ## setup environment + download models
#@markdown nên bật GPU session sẽ chạy nhanh hơn

%pip install -q ctranslate2

from tqdm import trange
import torch
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from ctranslate2 import Translator

chọn_model = "ngocdang83/HachimiMT-60-zh-vi"  # @param ["ngocdang83/HachimiMT-60-zh-vi", "DanVP/MoxhiMT-60"]

match chọn_model:
	case "ngocdang83/HachimiMT-60-zh-vi":
		_subfolder = "/ct2-int8_float32"
	case "DanVP/MoxhiMT-60":
		_subfolder = "/ct2-int8"
model_path = snapshot_download(chọn_model)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRANSLATOR = Translator(model_path + _subfolder, device=DEVICE)
TOKENIZER = AutoTokenizer.from_pretrained(chọn_model)

def dịch(text: list[str], batch_size: int) -> list[str]:
	inputs = [TOKENIZER.convert_ids_to_tokens(i) for i in TOKENIZER.encode(text, truncation=True)]
	outputs = TRANSLATOR.translate_batch(
		inputs,
		max_decoding_length=TOKENIZER.model_max_length,
		max_batch_size=batch_size,
		beam_size=4,
		no_repeat_ngram_size=2,
		repetition_penalty=1.2
	)
	results = TOKENIZER.decode(
		[TOKENIZER.convert_tokens_to_ids(i.hypotheses[0]) for i in outputs],
		skip_special_tokens=True
	)
	return results

các bước trên chỉ chạy 1 lần duy nhất trong 1 colab session

bước chạy convert dưới đây chạy bn lần cũng dc

up file text tiếng trung lên colab r chạy convert

chạy xong thì down file tiếng việt

In [ ]:
#@markdown ## chạy convert
file_raw  = "luanhoilacvien_cn.txt"  # @param {type: "string"}
file_convert = "luanhoilacvien_vi.txt"  # @param {type: "string"}
batch_size = 600  # @param {type: "number"}
#@markdown nếu bị tràn VRAM thì restart session và giảm batch size

lines = []
with open(file_raw, mode="r", encoding="utf-8") as f:
	for i in f.readlines():
		if (l := i.strip()) != "":
			lines.append(l)

with open(file_convert, mode="w", encoding="utf-8") as f:
	for i in trange(0, len(lines), batch_size):
		res = dịch(lines[i : i + batch_size], batch_size)
		f.write("\n".join(res) + "\n")